# kfp-nemo-curator-verify

[![Open in JupyterLab](https://img.shields.io/badge/Open%20in-JupyterLab-F37626?logo=jupyter&logoColor=white)](http://localhost:8888/lab/tree/git-miramar-labs-org/projects/kfp-nemo-curator-verify/notebook.ipynb)  [![Deploy to KFP](https://github.com/miramar-labs-org/kfp-nemo-curator-verify/actions/workflows/deploy-to-kfp.yaml/badge.svg)](https://github.com/miramar-labs-org/kfp-nemo-curator-verify/actions/workflows/deploy-to-kfp.yaml)  [![Undeploy from KFP](https://github.com/miramar-labs-org/kfp-nemo-curator-verify/actions/workflows/undeploy-from-kfp.yaml/badge.svg)](https://github.com/miramar-labs-org/kfp-nemo-curator-verify/actions/workflows/undeploy-from-kfp.yaml)

## Step development

- **All imports must be inside the function body** — each component runs in its own container
- Use `packages_to_install` on `@dsl.component` to add dependencies
- CPU steps: `base_image="python:3.11-slim"`
- GPU steps: `base_image="nvcr.io/nvidia/pytorch:26.04-py3"` + RAPIDS packages via `packages_to_install`
- GPU steps: call `.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("48G")` in the pipeline cell
- PVC `hf-model-cache` mounted at `/root/.cache/huggingface/`
- Data flow: `curator-input/{project}/raw/` → `extracted/` → `quality_filtered/` → `deduped/` → `curated/`

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Artifact

### preflight_check

Verifies the PVC input directory exists and contains source documents. Logs file count and
total bytes to MLflow. Fails fast with a clear message if `data_src/` was not staged.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-cpu:latest",
    packages_to_install=[],
)
def preflight_check(
    project_name: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import pathlib
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    raw_dir = pvc_root / "curator-input" / project_name / "raw"

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Input directory not found: {raw_dir}\n"
            "Run deploy_pipeline.py to stage data_src/ to the PVC first."
        )

    files = [f for f in raw_dir.rglob("*") if f.is_file()]
    if not files:
        raise ValueError(
            f"No files found in {raw_dir} — add documents to data_src/ and redeploy."
        )

    total_bytes = sum(f.stat().st_size for f in files)
    exts = {f.suffix.lower() for f in files}

    # Verify all stage output dirs are writable — KFP pods run as uid=0 with
    # capabilities.drop:[ALL], removing DAC_OVERRIDE. Without it, even root cannot
    # write to dirs owned by another uid with mode 755. deploy_pipeline.py pre-creates
    # these dirs and chmod 0o777's them; this block fails fast if that was skipped.
    stage_dirs = ["extracted", "quality_filtered", "deduped", "curated"]
    for subdir in stage_dirs:
        d = pvc_root / "curator-input" / project_name / subdir
        if not d.exists():
            raise FileNotFoundError(
                f"Stage dir missing: {d}\n"
                "Re-run deploy_pipeline.py — it pre-creates all stage dirs with chmod 777."
            )
        probe = d / ".write_probe"
        try:
            probe.write_text("ok")
            probe.unlink()
        except PermissionError as e:
            raise PermissionError(
                f"Stage dir not writable by container: {d}\n"
                f"Fix: chmod -R a+w {pvc_root}/curator-input/{project_name}/"
            ) from e

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/preflight_check"):
        mlflow.log_metric("stage/input_file_count", len(files))
        mlflow.log_metric("stage/input_total_bytes", total_bytes)
        mlflow.log_param("stage/input_extensions", ",".join(sorted(exts)))

    print(f"Preflight OK: {len(files)} file(s), {total_bytes / 1024:.1f} KB, types={exts}")

### extract_text

Reads raw files from the PVC, applies Unicode normalization and boilerplate removal,
writes one JSON record per document to `extracted/docs.jsonl`.
Supports `.txt`, `.md`, `.html`, and `.jsonl` input formats.

See `WORKBOOK.md` for implementation patterns per format.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-cpu:latest",
    packages_to_install=[],
)
def extract_text(
    project_name: str,
    run_id: str,
    input_format: str,
    input_text_field: str,
    input_id_field: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import pathlib
    import json
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    in_dir = pvc_root / "curator-input" / project_name / "raw"
    out_dir = pvc_root / "curator-input" / project_name / "extracted"
    out_dir.mkdir(parents=True, exist_ok=True)

    import re
    import ftfy
    files = sorted([f for f in in_dir.rglob("*") if f.is_file()])
    n_in = len(files)
    n_empty = 0
    records = []
    for f in files:
        raw = f.read_text(encoding="utf-8", errors="replace")
        if f.suffix.lower() == ".jsonl":
            for i, line in enumerate(raw.splitlines()):
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    n_empty += 1
                    continue
                text = ftfy.fix_text(rec.get(input_text_field, "")).strip()
                text = re.sub(r"\s+", " ", text)
                doc_id = str(rec.get(input_id_field, f"{f.stem}-{i}"))
                if text:
                    records.append({"id": doc_id, "text": text, "source": f.name})
                else:
                    n_empty += 1
        else:
            text = ftfy.fix_text(raw).strip()
            text = re.sub(r"\s+", " ", text)
            if text:
                records.append({"id": f.stem, "text": text, "source": f.name})
            else:
                n_empty += 1
    (out_dir / "docs.jsonl").write_text("\n".join(json.dumps(r) for r in records))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/extract_text"):
        mlflow.log_metric("stage/extract_text/docs_in", n_in)
        mlflow.log_metric("stage/extract_text/docs_out", len(records))
        mlflow.log_metric("stage/extract_text/empty_skipped", n_empty)

    print(f"extract_text: {n_in} in → {len(records)} out ({n_empty} empty skipped)")

### quality_filter

GPU component. Applies NeMo Curator heuristic quality filters using the cuDF backend.
Removes documents that are too short/long, have too many symbols, or too few stop words.

Installs `nemo-curator[cuda12x]` from `https://pypi.nvidia.com` — requires arm64 RAPIDS wheels.
See `WORKBOOK.md → GPU wheel fallback` if installation fails.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-gpu:latest",
    packages_to_install=[],
)
def quality_filter(
    project_name: str,
    run_id: str,
    min_doc_length: int,
    max_doc_length: int,
    min_mean_word_length: float,
    max_mean_word_length: float,
    max_symbol_to_word_ratio: float,
    min_stop_word_fraction: float,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import pathlib
    import json
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    in_dir = pvc_root / "curator-input" / project_name / "extracted"
    out_dir = pvc_root / "curator-input" / project_name / "quality_filtered"
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        import cudf
        _use_gpu = True
        print(f"cuDF {cudf.__version__} available — GPU backend")
    except ImportError as e:
        _use_gpu = False
        print(f"cuDF not available ({e}) — CPU fallback")
    import re
    _lines = [l for l in (in_dir / "docs.jsonl").read_text().splitlines() if l.strip()]
    docs_in = len(_lines)
    _recs = [json.loads(l) for l in _lines]
    if _use_gpu:
        _df = cudf.DataFrame({
            "id": [r["id"] for r in _recs],
            "text": [r["text"] for r in _recs],
            "source": [r["source"] for r in _recs],
        })
        _wc = _df["text"].str.split().list.len()
        _tl = _df["text"].str.len()
        _mwl = (_tl / _wc.clip(lower=1)).fillna(0.0)
        _text_alnum = _df["text"].str.replace(r"[^a-zA-Z0-9 ]", "", regex=True).str.replace(r"  +", " ", regex=True).str.strip()
        _wc_alpha = _text_alnum.str.split().list.len()
        _sc = (_wc - _wc_alpha).clip(lower=0)
        _sr = (_sc / _wc.clip(lower=1)).fillna(0.0)
        _mask = (
            (_wc >= min_doc_length) & (_wc <= max_doc_length) &
            (_mwl >= min_mean_word_length) & (_mwl <= max_mean_word_length) &
            (_sr <= max_symbol_to_word_ratio)
        )
        _kept = _df[_mask][["id", "text", "source"]].to_pandas().to_dict("records")
    else:
        _STOP = {"the", "be", "to", "of", "and", "a", "in", "that", "have", "it"}
        def _ok(r):
            ws = r["text"].split()
            wc = len(ws)
            if not (min_doc_length <= wc <= max_doc_length):
                return False
            mwl = sum(len(w) for w in ws) / max(wc, 1)
            if not (min_mean_word_length <= mwl <= max_mean_word_length):
                return False
            sc = sum(1 for w in ws if w and not any(c.isalnum() for c in w))
            if sc / max(wc, 1) > max_symbol_to_word_ratio:
                return False
            sw = sum(1 for w in ws if w.lower() in _STOP) / max(wc, 1)
            return sw >= min_stop_word_fraction
        _kept = [r for r in _recs if _ok(r)]
    docs_out = len(_kept)
    (out_dir / "docs.jsonl").write_text("\n".join(json.dumps(r) for r in _kept))

    rejection_rate = 1.0 - (docs_out / docs_in) if docs_in > 0 else 0.0
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/quality_filter"):
        mlflow.log_metric("stage/quality_filter/docs_in", docs_in)
        mlflow.log_metric("stage/quality_filter/docs_out", docs_out)
        mlflow.log_metric("stage/quality_filter/rejection_rate", rejection_rate)

    print(f"quality_filter: {docs_in} in → {docs_out} out ({rejection_rate:.1%} rejected)")

### deduplication

GPU component. Runs exact MD5 dedup first, then fuzzy MinHash LSH dedup via cuDF.
Both stages use the RAPIDS cuDF backend for accelerated dataframe operations.

See `WORKBOOK.md` for the NeMo Curator `ExactDuplicates` and `FuzzyDuplicates` patterns.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-gpu:latest",
    packages_to_install=[],
)
def deduplication(
    project_name: str,
    run_id: str,
    fuzzy_jaccard_threshold: float,
    fuzzy_ngram_size: int,
    fuzzy_num_hashes: int,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import pathlib
    import json
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    in_dir = pvc_root / "curator-input" / project_name / "quality_filtered"
    out_dir = pvc_root / "curator-input" / project_name / "deduped"
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        import cudf
        print(f"cuDF {cudf.__version__} available — GPU dedup active")
    except ImportError as e:
        print(f"cuDF not available ({e}) — CPU dedup fallback")
    import hashlib
    _lines = [l for l in (in_dir / "docs.jsonl").read_text().splitlines() if l.strip()]
    docs_in = len(_lines)
    _recs = [json.loads(l) for l in _lines]
    _seen = set()
    _after_exact = []
    for r in _recs:
        _h = hashlib.md5(r["text"].encode()).hexdigest()
        if _h not in _seen:
            _seen.add(_h)
            _after_exact.append(r)
    exact_removed = docs_in - len(_after_exact)
    def _ngrams(text, n):
        toks = text.lower().split()
        return set(tuple(toks[i:i + n]) for i in range(max(0, len(toks) - n + 1)))
    def _minhash(ngs, k):
        return [min((hash((s, g)) & 0xFFFFFFFF for g in ngs), default=0) for s in range(k)]
    _sigs = [_minhash(_ngrams(r["text"], fuzzy_ngram_size), fuzzy_num_hashes) for r in _after_exact]
    _nb = max(1, fuzzy_num_hashes // 4)
    _rpb = max(1, fuzzy_num_hashes // _nb)
    _dups = set()
    for _b in range(_nb):
        _bkts = {}
        for _i, _sig in enumerate(_sigs):
            if _i in _dups:
                continue
            _key = tuple(_sig[_b * _rpb:(_b + 1) * _rpb])
            if _key in _bkts:
                _j = _bkts[_key]
                _ni = _ngrams(_after_exact[_i]["text"], fuzzy_ngram_size)
                _nj = _ngrams(_after_exact[_j]["text"], fuzzy_ngram_size)
                _u = _ni | _nj
                if _u and len(_ni & _nj) / len(_u) >= fuzzy_jaccard_threshold:
                    _dups.add(_i)
            else:
                _bkts[_key] = _i
    _kept = [r for _i, r in enumerate(_after_exact) if _i not in _dups]
    fuzzy_removed = len(_after_exact) - len(_kept)
    docs_out = len(_kept)
    (out_dir / "docs.jsonl").write_text("\n".join(json.dumps(r) for r in _kept))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/deduplication"):
        mlflow.log_metric("stage/deduplication/docs_in", docs_in)
        mlflow.log_metric("stage/deduplication/exact_removed", exact_removed)
        mlflow.log_metric("stage/deduplication/fuzzy_removed", fuzzy_removed)
        mlflow.log_metric("stage/deduplication/docs_out", docs_out)

    print(f"deduplication: {docs_in} in → {docs_out} out "
          f"(exact={exact_removed}, fuzzy={fuzzy_removed} removed)")

### pii_redaction

CPU component. Detects and redacts PII entities using presidio + spaCy NER.
Writes the final curated dataset to `curated/docs.jsonl`.

See `WORKBOOK.md` for NeMo Curator `PiiModifier` and direct presidio patterns.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-cpu:latest",
    packages_to_install=[],
)
def pii_redaction(
    project_name: str,
    run_id: str,
    pii_entities: str,
    pii_action: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import pathlib
    import json
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    in_dir = pvc_root / "curator-input" / project_name / "deduped"
    out_dir = pvc_root / "curator-input" / project_name / "curated"
    out_dir.mkdir(parents=True, exist_ok=True)

    entities = json.loads(pii_entities)

    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"],
        check=True,
    )
    from presidio_analyzer import AnalyzerEngine
    from presidio_anonymizer import AnonymizerEngine
    _analyzer = AnalyzerEngine()
    _anonymizer = AnonymizerEngine()
    _lines = [l for l in (in_dir / "docs.jsonl").read_text().splitlines() if l.strip()]
    _recs = [json.loads(l) for l in _lines]
    n_docs = len(_recs)
    n_pii = 0
    _output = []
    for r in _recs:
        text = r["text"]
        try:
            _res = _analyzer.analyze(text=text, entities=entities, language="en")
            n_pii += len(_res)
            if _res:
                _anon = _anonymizer.anonymize(text=text, analyzer_results=_res)
                r = {**r, "text": _anon.text}
        except Exception as _exc:
            print(f"PII error on doc {r.get('id', '?')}: {_exc}")
        _output.append(r)
    (out_dir / "docs.jsonl").write_text("\n".join(json.dumps(r) for r in _output))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/pii_redaction"):
        mlflow.log_metric("stage/pii_redaction/docs_processed", n_docs)
        mlflow.log_metric("stage/pii_redaction/pii_instances_found", n_pii)
        mlflow.log_param("stage/pii_redaction/entities", ",".join(entities))
        mlflow.log_param("stage/pii_redaction/action", pii_action)

    print(f"pii_redaction: {n_docs} docs processed, {n_pii} PII instances redacted")

### curator_report

Reads the final curated dataset, computes summary statistics, logs to MLflow,
and writes `curation_report.json` to the output PVC directory.

In [ ]:
@dsl.component(
    base_image="ghcr.io/miramar-labs-org/kfp-base-cpu:latest",
    packages_to_install=[],
)
def curator_report(
    project_name: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
) -> None:
    import json
    import pathlib
    import mlflow

    pvc_root = pathlib.Path("/root/.cache/huggingface")
    curated_dir = pvc_root / "curator-input" / project_name / "curated"
    out_dir = pvc_root / "curator-output" / project_name / run_id
    out_dir.mkdir(parents=True, exist_ok=True)

    curated_file = curated_dir / "docs.jsonl"
    if not curated_file.exists():
        raise FileNotFoundError(
            f"Curated dataset not found: {curated_file}\n"
            "pii_redaction must complete successfully first."
        )

    docs = [
        json.loads(line)
        for line in curated_file.read_text().splitlines()
        if line.strip()
    ]
    total_docs = len(docs)
    total_chars = sum(len(d.get("text", "")) for d in docs)
    total_words = sum(len(d.get("text", "").split()) for d in docs)
    mean_doc_length = total_chars / total_docs if total_docs else 0.0

    report = {
        "run_id": run_id,
        "project_name": project_name,
        "curated_doc_count": total_docs,
        "total_chars": total_chars,
        "total_words": total_words,
        "mean_doc_length_chars": round(mean_doc_length, 1),
        "output_path": str(curated_file),
    }
    (out_dir / "curation_report.json").write_text(json.dumps(report, indent=2))

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}/curator_report"):
        mlflow.log_metric("final/curated_doc_count", total_docs)
        mlflow.log_metric("final/total_chars", total_chars)
        mlflow.log_metric("final/total_words", total_words)
        mlflow.log_metric("final/mean_doc_length_chars", mean_doc_length)
        mlflow.log_param("output_path", str(curated_file))
        mlflow.log_dict(report, "curation_report.json")

    print(f"Curator report: {total_docs} docs, {total_words:,} words → {out_dir}/curation_report.json")

### Pipeline

Wires all 6 steps together. Config values are read from `config.yaml` at build time and
become pipeline parameter defaults. GPU components request `nvidia.com/gpu`.
All tasks mount `hf-model-cache` and receive secrets from `mlabs-api-keys`.

In [ ]:
import json as _json
import yaml as _yaml
import pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_pipeline_name = "kfp-nemo-curator-verify"

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="kfp-nemo-curator-verify")
def pipeline(
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "kfp-nemo-curator-verify",
    input_format: str = _cfg["input"]["format"],
    input_text_field: str = _cfg["input"]["text_field"],
    input_id_field: str = _cfg["input"].get("id_field", "id"),
    min_doc_length: int = _cfg["quality_filter"]["min_doc_length"],
    max_doc_length: int = _cfg["quality_filter"]["max_doc_length"],
    min_mean_word_length: float = _cfg["quality_filter"]["min_mean_word_length"],
    max_mean_word_length: float = _cfg["quality_filter"]["max_mean_word_length"],
    max_symbol_to_word_ratio: float = _cfg["quality_filter"]["max_symbol_to_word_ratio"],
    min_stop_word_fraction: float = _cfg["quality_filter"]["min_stop_word_fraction"],
    fuzzy_jaccard_threshold: float = _cfg["deduplication"]["fuzzy_jaccard_threshold"],
    fuzzy_ngram_size: int = _cfg["deduplication"]["fuzzy_ngram_size"],
    fuzzy_num_hashes: int = _cfg["deduplication"]["fuzzy_num_hashes"],
    pii_entities: str = _json.dumps(_cfg["pii"]["entities"]),
    pii_action: str = _cfg["pii"]["action"],
):
    preflight = preflight_check(
        project_name=_pipeline_name,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )

    extract = extract_text(
        project_name=_pipeline_name,
        run_id=run_id,
        input_format=input_format,
        input_text_field=input_text_field,
        input_id_field=input_id_field,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    extract.after(preflight)

    qual = quality_filter(
        project_name=_pipeline_name,
        run_id=run_id,
        min_doc_length=min_doc_length,
        max_doc_length=max_doc_length,
        min_mean_word_length=min_mean_word_length,
        max_mean_word_length=max_mean_word_length,
        max_symbol_to_word_ratio=max_symbol_to_word_ratio,
        min_stop_word_fraction=min_stop_word_fraction,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    qual.after(extract)
    qual.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("48G")

    dedup = deduplication(
        project_name=_pipeline_name,
        run_id=run_id,
        fuzzy_jaccard_threshold=fuzzy_jaccard_threshold,
        fuzzy_ngram_size=fuzzy_ngram_size,
        fuzzy_num_hashes=fuzzy_num_hashes,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    dedup.after(qual)
    dedup.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("48G")

    pii = pii_redaction(
        project_name=_pipeline_name,
        run_id=run_id,
        pii_entities=pii_entities,
        pii_action=pii_action,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    pii.after(dedup)

    report = curator_report(
        project_name=_pipeline_name,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    report.after(pii)

    _PVC = "hf-model-cache"
    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = {"OPENAI_API_KEY": "OPENAI_API_KEY", "HF_TOKEN": "HF_TOKEN"}
    for _task in [preflight, extract, qual, dedup, pii, report]:
        k8s_ext.mount_pvc(_task, pvc_name=_PVC, mount_path="/root/.cache/huggingface")
        k8s_ext.use_secret_as_env(_task, _SECRET, _SECRET_KEYS)

    # curator_report uploads artifacts to MinIO via boto3 — needs S3-compatible creds
    _ARTIFACT_SECRET = "mlflow-artifact-env"
    _ARTIFACT_KEYS = {k: k for k in [
        "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION",
        "AWS_S3_FORCE_PATH_STYLE", "MLFLOW_S3_ENDPOINT_URL", "MLFLOW_S3_IGNORE_TLS",
    ]}
    k8s_ext.use_secret_as_env(report, _ARTIFACT_SECRET, _ARTIFACT_KEYS)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell to regenerate `pipeline.py`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8890:localhost:8890 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8890")
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
    experiment_name="kfp-nemo-curator-verify",
)
print(f"Run submitted: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.run.status
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED"):
        break
    time.sleep(30)